In [1]:
from thefuzz import fuzz
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv
from openai import OpenAI
import numpy as np
import pandas as pd
import re
import warnings

warnings.filterwarnings("ignore")

load_dotenv()

c:\Users\User\Desktop\Logistics-RPF\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

### Target Schema Fields

In [2]:
schema = [
    "origin_port",
    "destination_port",
    "container_type_20gp_rate",
    "container_type_40hq_rate",
    "estimated_time_of_departure",
    "transit_time_days",
    "currency",
]

schema

['origin_port',
 'destination_port',
 'container_type_20gp_rate',
 'container_type_40hq_rate',
 'estimated_time_of_departure',
 'transit_time_days',
 'currency']

In [3]:
# data\input

df = pd.read_csv(r"..\data\input\rfp_standard.csv")
df.head()

,Orig Port,Destination,20GP,40HQ Rate,ETD,Transit (days),Currency
0,Shanghai,Rotterdam,1200,2200,2024-03-15,28,USD
1,Ningbo,Hamburg,1300,2400,2024-03-18,32,USD
2,Shenzhen,Antwerp,1250,2300,2024-03-20,30,EUR


In [4]:
columns = df.columns.tolist()
print(columns)

['Orig Port', 'Destination', '20GP', '40HQ Rate', 'ETD', 'Transit (days)', 'Currency']


### Calculate Similarity Scores using Fuzzy Matching

In [5]:
for target_col, input_col in zip(schema, columns):
    score = fuzz.ratio(target_col, input_col)
    print(f"Similarity Score: {score}") 

Similarity Score: 60
Similarity Score: 74
Similarity Score: 14
Similarity Score: 30
Similarity Score: 0
Similarity Score: 65
Similarity Score: 88


###  Normalizing the column names + fuzzy matching

In [6]:
for target_col, input_col in zip(schema, columns):
    score = fuzz.ratio(target_col.lower().replace(" ", ""), input_col.lower().replace("_", ""))
    print(f"Similarity Score: {score}")
    

Similarity Score: 80
Similarity Score: 81
Similarity Score: 29
Similarity Score: 48
Similarity Score: 20
Similarity Score: 71
Similarity Score: 100


**Observation:** Increase in similarity score after normalization.

In [65]:
def preprocess_columns(columns):
    # regex to standardize column names
    # replace spaces with underscores
    columns = [col.replace(' ', '_') for col in columns]
    # remove special characters
    columns = [re.sub(r'[^A-Za-z0-9_]+', '', col) for col in columns]
    # convert to lowercase
    columns = [col.lower() for col in columns]
    
    return columns  

processed_cols = preprocess_columns(columns)
#print(processed_cols)

for target_col, input_col in zip(schema, processed_cols):
    score = fuzz.ratio(target_col.lower().replace(" ", ""), input_col)
    print(f"Similarity Score: {score}")

Similarity Score: 90
Similarity Score: 81
Similarity Score: 29
Similarity Score: 55
Similarity Score: 20
Similarity Score: 83
Similarity Score: 100


### Sentence Tranformer Model from Hugging Face 

In [31]:
model = SentenceTransformer("all-MiniLM-L6-v2")

for target_col, input_col in zip(schema, processed_cols):
    query_vec  = model.encode([target_col.lower()])
    fields_vec = model.encode([input_col])
    scores = cosine_similarity(query_vec, fields_vec)[0][0] *100
    print(f"{target_col:<25} | {input_col:<25} : {scores:.2f}")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14533.18it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


origin_port               | orig_port                 : 68.47
destination_port          | destination               : 52.82
container_type_20gp_rate  | 20gp                      : 58.54
container_type_40hq_rate  | 40hq_rate                 : 73.25
estimated_time_of_departure | etd                       : 14.15
transit_time_days         | transit_days              : 95.73
currency                  | currency                  : 100.00


**Observation:** performs well on direct matches but struggles with abbreviations and synonyms.

In [36]:
def hybrid_score(target_col, input_col, weight_semantic=0.7):

    query_vec  = model.encode([target_col.lower()])
    fields_vec = model.encode([input_col])
    semantic_score = float(cosine_similarity(query_vec, fields_vec)[0][0]) *100
    
    fuzzy_score = fuzz.ratio(target_col.lower(), input_col)
    
    return semantic_score, fuzzy_score, (weight_semantic * semantic_score) + ((1 - weight_semantic) * fuzzy_score)

for target_col, input_col in zip(schema, processed_cols):
    
    print(f"{target_col:<25} | {input_col:<25} : {hybrid_score(target_col, input_col, weight_semantic=0.7)}")

origin_port               | orig_port                 : (68.47333908081055, 90, 74.93133735656738)
destination_port          | destination               : (52.81971096992493, 81, 61.27379767894745)
container_type_20gp_rate  | 20gp                      : (58.537471294403076, 29, 49.676229906082156)
container_type_40hq_rate  | 40hq_rate                 : (73.24714660644531, 55, 67.77300262451172)
estimated_time_of_departure | etd                       : (14.149399101734161, 20, 15.904579371213913)
transit_time_days         | transit_days              : (95.72909474372864, 83, 91.91036632061005)
currency                  | currency                  : (100.00002384185791, 100, 100.00001668930054)


### Similarity using OpenAI Embedding Model

In [37]:
client = OpenAI()

def normalize(text: str) -> str:
    text = text.replace("_", " ").replace("-", " ")
    text = re.sub(r'([A-Z0-9])', r' \1', text)
    return text.lower().strip()

def enrich(text: str, domain="shipping logistics") -> str:
    return f"{domain} field: {normalize(text)}"

def get_embedding(text: str, model="text-embedding-3-large"):
    text = text.replace("\n", " ").strip()
    return client.embeddings.create(input=[text], model=model).data[0].embedding

def similarity_score(a: str, b: str) -> float:
    vec_a = get_embedding(enrich(a))
    vec_b = get_embedding(enrich(b))
    score = cosine_similarity([vec_a], [vec_b])[0][0]
    return round(float(score), 4)


for target_col, input_col in zip(schema, columns):
    print(f"{target_col:<25} | {input_col:<25} : {similarity_score(target_col, input_col):.2f}")

origin_port               | Orig Port                 : 0.93
destination_port          | Destination               : 0.92
container_type_20gp_rate  | 20GP                      : 0.79
container_type_40hq_rate  | 40HQ Rate                 : 0.85
estimated_time_of_departure | ETD                       : 0.82
transit_time_days         | Transit (days)            : 0.96
currency                  | Currency                  : 1.00


In [38]:
similarity_score("currency", "ccy")

0.9079

In [39]:
similarity_score("origin_port", "POL")

0.7486

**Observation:** OpenAI embeddings perform better than Sentence Transformers, but has difficulty with abbrevations.

### Create Knowledge Base for mapping

In [66]:
FIELD_KNOWLEDGE_BASE = {
    "origin_port": [
        "origin port", "port of loading", "load port", "loading port", # "pol",
        "departure port", "source port", "from port", "shipper port",
        "port of origin", "place of receipt", "por"
    ],
    "destination_port": [
        "destination port", "discharge port", "pod", "port of discharge",
        "delivery port", "arrival port", "to port", "consignee port",
        "destination", "place of delivery", "final port", "disport"
    ],
    "container_type_20gp_rate": [
        "20gp", "20 gp", "20ft", "20 ft", "20 foot", "twenty foot",
        "20gp rate", "20 gp rate", "teu rate", "teu", "twenty equivalent unit",
        "20 general purpose", "20gp freight", "20 dry", "rate 20"
    ],
    "container_type_40hq_rate": [
        "40hq", "40 hq", "40ft hq", "40 high cube", "40hc", "40 hc",
        "40hq rate", "40 hq rate", "high cube rate", "40 high", "feu rate",
        "40hq freight", "40 cube", "rate 40hq", "forty high cube"
    ],
    "estimated_time_of_departure": [
        "etd", "estimated departure", "departure date", "sailing date",
        "vessel departure", "expected departure", "cutoff date", "sail date",
        "scheduled departure", "departure time", "vessel etd", "est departure"
    ],
    "transit_time_days": [
        "transit time", "transit days", "tt", "sailing time", "travel time",
        "voyage days", "journey time", "shipping days", "days in transit",
        "lead time", "delivery days", "transit period", "number of days"
    ],
    "currency": [
        "currency", "curr", "ccy", "currency code", "payment currency",
        "freight currency", "rate currency", "usd", "eur", "inr",
        "denomination", "monetary unit", "fx", "forex"
    ]
}

In [78]:
def normalize(text: str) -> str:
    text = text.replace("_", " ").replace("-", " ")
    text = re.sub(r'(?<=[a-z])(?=[A-Z])', ' ', text)   # camelCase → camel Case    
    text = re.sub(r'(?<=[A-Za-z])(?=\d)', ' ', text)   # letters→digits: "20gp" stays, "type20" → "type 20"     # digits→letters: "20gp" → "20 gp"    
    text = re.sub(r'\s+', ' ', text)                   # collapse spaces     
    return text.lower().strip()

def match_with_kb(input_field: str, threshold=70) -> list:
    normalized_input = normalize(input_field)
    results = []
    
    for schema_field, synonyms in FIELD_KNOWLEDGE_BASE.items():
        best_score = 0

        for synonym in synonyms:
            score = fuzz.ratio(normalized_input, synonym)
            best_score = max(best_score, score)
           
            # Exact match short circuit
            if normalized_input == synonym:
                best_score = 100
                break

        if best_score >= threshold:
            results.append({
                "schema_field": schema_field,
                "input_field":  input_field,
                "score":        best_score,
                "status":       "Match" if best_score == 100 else "Verify"
            })  
    
    if len(results) == 0:
        results.append({
            "schema_field": "Unknown",
            "input_field":  input_field,
            "score": 0,
            "status": "Verify"
        })

    results.sort(key=lambda x: x["score"], reverse=True)
    return results


In [79]:
column_names = ["POL", "POD", "40HQ Rate", "Transit (days)", "ETD", "Currency"]

for col in column_names:
    print(match_with_kb(col))

[{'schema_field': 'Unknown', 'input_field': 'POL', 'score': 0, 'status': 'Verify'}]
[{'schema_field': 'destination_port', 'input_field': 'POD', 'score': 100, 'status': 'Match'}]
[{'schema_field': 'container_type_40hq_rate', 'input_field': '40HQ Rate', 'score': 100, 'status': 'Match'}]
[{'schema_field': 'transit_time_days', 'input_field': 'Transit (days)', 'score': 92, 'status': 'Verify'}]
[{'schema_field': 'estimated_time_of_departure', 'input_field': 'ETD', 'score': 100, 'status': 'Match'}]
[{'schema_field': 'currency', 'input_field': 'Currency', 'score': 100, 'status': 'Match'}]


**Observation:** Using a stored knowledge map + normalization + fuzzy match, seems to perform well.

### OpenAI Chat model with tailored prompt

In [60]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
import json
import re

llm = ChatOpenAI(model="gpt-4o")

prompt_template = """
You are an expert in international shipping and freight forwarding. Your task is to map non-standard RFP (Request for Proposal) column headers to a standardized internal schema.

**Target Schema:**
- origin_port
- destination_port
- container_type_20gp_rate
- container_type_40hq_rate
- estimated_time_of_departure
- transit_time_days
- currency

**Input Columns:** {input_columns}

**Instructions:**
1. Analyze the input column names.
2. Determine which target schema fields they most likely represent.
3. Provide the reasoning for the mapping based on domain knowledge.
4. Provide the mapping in the following JSON format:
   {{
    "<input_column_name>":
        {{
            "schema_column": "<field_name>",
            "confidence": <score_0_to_1>,
            "reasoning": "<reasoning>"
        }}
   }}

**Scoring Guidelines:**
- 0.9-1.0: Direct match or very strong semantic similarity
- 0.7-0.89: Good match with some ambiguity
- 0.5-0.69: Possible match, requires context
- Below 0.5: Unlikely match, flag for manual review

**Examples:**
Input: "40HQ Rate" -> {{"40HQ Rate": {{"schema_column": "container_type_40hq_rate", "confidence": 0.92, "reasoning": "Direct match"}}}}
Input: "Transit (days)" -> {{"Transit (days)": {{"schema_column": "transit_time_days", "confidence": 0.88, "reasoning": "Direct match"}}}}

Now, map the columns: {input_columns}
"""

prompt = PromptTemplate(
    input_variables=["input_columns"],
    template=prompt_template
)

llm_chain = prompt | llm

# Pass a list of column names
column_names = ["POL", "POD", "40HQ Rate", "ETD", "Currency"]
result = llm_chain.invoke({"input_columns": ", ".join(column_names)})

# Strip markdown code blocks if present and parse JSON
raw = re.sub(r"```json|```", "", result.content).strip()
mapping = json.loads(raw)

print(json.dumps(mapping, indent=4))

{
    "POL": {
        "schema_column": "origin_port",
        "confidence": 0.95,
        "reasoning": "POL stands for 'Port of Loading' which is synonymous with 'origin port'. The terms are standard in shipping industry terminology."
    },
    "POD": {
        "schema_column": "destination_port",
        "confidence": 0.95,
        "reasoning": "POD stands for 'Port of Discharge' which directly corresponds to 'destination port'. This is a well-known term in shipping."
    },
    "40HQ Rate": {
        "schema_column": "container_type_40hq_rate",
        "confidence": 0.92,
        "reasoning": "The term '40HQ' designates a 40-foot High-Cube container, hence '40HQ Rate' directly maps to 'container_type_40hq_rate'."
    },
    "ETD": {
        "schema_column": "estimated_time_of_departure",
        "confidence": 0.9,
        "reasoning": "ETD stands for 'Estimated Time of Departure'. It is a common abbreviation used in logistics to indicate the scheduled departure time."
    },
    "C

In [62]:
col = ['Transit (days)']

result = llm_chain.invoke({"input_columns": ", ".join(col)})

print(result.content)

```json
{
    "Transit (days)": {
        "schema_column": "transit_time_days",
        "confidence": 0.95,
        "reasoning": "The term 'Transit (days)' directly indicates the duration of transit between the origin and destination, which aligns with 'transit_time_days' in the target schema. The specificity of 'days' further solidifies this as a representation of time, making it a very strong match."
    }
}
```


#### Observation:

- The LLM model is able to map the columns to the target schema with high confidence based on Domain knowledge, but requires API calls which can be expensive.
- A knowledge base of common abbreviations and synonyms can be used to improve the accuracy of the mapping, and works well when used along with fuzzy matching and normalization techniques.

In [2]:
import pandas as pd

In [5]:
CURRENCIES = {"USD", "EUR", "GBP", "JPY", "CNY", "AUD", "CAD", "CHF", "HKD", "SGD"}

In [49]:
df = pd.read_csv(r"..\data\input\rfp_currency_in_data.csv")
df.head()

,Origin,Dest Port,20GP Rate,40HC Rate,Sailing Date Transit Time,T/T,Curr
0,Shanghai,Rotterdam,1200 USD,2200 USD,2024-03-15,28,USD
1,Ningbo,Hamburg,1300 USD,2400 USD,2024-03-18,32,USD
2,Shenzhen,Antwerp,1250 EUR,2300 EUR,2024-03-20,30,EUR
3,Qingdao,Felixstowe,1150 GBP,2100 GBP,2024-03-22,35,GBP


In [51]:
for col in df1.columns[:-1]:
    split = df1[col].astype(str).str.split()

    first = split.str[0]
    last = split.str[-1]

    mask = last.isin(CURRENCIES)

    df1.loc[mask, col] = first

In [52]:
df1

,Origin,Dest Port,20GP Rate,40HC Rate,Sailing Date Transit Time,T/T,Curr
0,Shanghai,Rotterdam,1200,2200,2024-03-15,28,USD
1,Ningbo,Hamburg,1300,2400,2024-03-18,32,USD
2,Shenzhen,Antwerp,1250,2300,2024-03-20,30,EUR
3,Qingdao,Felixstowe,1150,2100,2024-03-22,35,GBP
